# Phase 2: Temporal Reconstruction Test
### **Objective:** Verify that the `MotorLoader` (or any other ingestion classes) successfully passes data to the `PhaseSpaceEmbedder`, and visually confirm that Takens' Theorem unfolds the 1D motor signal into a multi-dimensional phase-space trajectory.
***

# Cell 1: Environment Setup & Cloud Mounting

In [7]:
import sys
import os

# 1. Mount Google Drive
try:
    from google.colab import drive
    print("[INFO] Mounting Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("❌ Error: Not running in a Colab environment!")

# 2. Inject your specific repo path
project_root = '/content/drive/MyDrive/CARDD-Tech-Diagnostic-Pipeline'
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"✅ Cloud path injected: {project_root}")

# 3. Force module autoreloading
try:
    import IPython
    shell = IPython.get_ipython()
    if shell is not None:
        shell.run_line_magic('load_ext', 'autoreload')
        shell.run_line_magic('autoreload', '2')
except Exception as e:
    pass

[INFO] Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


***
# Cell 2: Data Ingestion Pipeline (Phase 1)

In [8]:
from src.ingestion.motor_loader import MotorDataLoader

print("[INFO] Ingesting raw motor dataset...")
loader = MotorDataLoader(data_path=f"{project_root}/data/raw_motor_signals.csv" ,
                         sensor_cols=['Va', 'Vb', 'Vc', 'Ia', 'Ib', 'Ic'])
loader.load_and_clean()
X_np, y_np = loader.get_numpy()

# We will extract just the very first sensor channel (e.g., Voltage Phase U) 
# to keep the visualization clean. We'll look at the first 2000 samples.
signal_1d = X_np[:2000, 0] 

print(f"✅ Extracted 1D test signal. Shape: {signal_1d.shape}")

[INFO] Ingesting raw motor dataset...
✅ Extracted 1D test signal. Shape: (2000,)


***
# Cell 3: Phase Space Reconstruction (Phase 2)

In [ ]:
!pip install -q nolds
from src.features.delay_embed import PhaseSpaceEmbedder

print("[INFO] Initializing TISEAN-alternative Embedder...")

# We will force m=3 just so we can plot it easily in a 3D scatter plot.
# We will let the algorithm automatically calculate the optimal delay (tau).
embedder = PhaseSpaceEmbedder(m=3) 

print("[INFO] Folding 1D signal into Phase Space...")
X_embedded = embedder.fit_transform(signal_1d)
_, tau = embedder.fitted_params_

print(f"✅ Embedding complete!")
print(f"   -> Optimal Time Delay (tau) calculated: {tau}")
print(f"   -> Embedded Matrix Shape: {X_embedded.shape}")

ModuleNotFoundError: No module named 'nolds'

***
# Cell 4: Visual Verification

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Set up a professional plot layout
fig = plt.figure(figsize=(16, 6))
plt.suptitle(f"Takens' Theorem: 1D Signal => 3D Phase Space ($\\tau={tau}$)", fontsize=16)

# Subplot 1: The original 1D Signal
ax1 = fig.add_subplot(121)
ax1.plot(signal_1d, color='blue', alpha=0.7)
ax1.set_title("Original 1D Sensor Stream", fontsize=12)
ax1.set_xlabel("Time Step")
ax1.set_ylabel("Normalized Amplitude")
ax1.grid(True, linestyle='--', alpha=0.5)

# Subplot 2: The 3D Phase Space Reconstruction
ax2 = fig.add_subplot(122, projection='3d')
x_coords = X_embedded[:, 0] # t
y_coords = X_embedded[:, 1] # t + tau
z_coords = X_embedded[:, 2] # t + 2*tau

# Use a color map based on time so you can see the trajectory flow
scatter = ax2.scatter(x_coords, y_coords, z_coords, 
                      c=np.arange(len(x_coords)), cmap='viridis', 
                      s=2, alpha=0.8)

ax2.set_title("Reconstructed 3D Attractor", fontsize=12)
ax2.set_xlabel("x(t)")
ax2.set_ylabel(f"x(t + {tau})")
ax2.set_zlabel(f"x(t + 2{tau})")

# Clean up layout and display
plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()